# Wan 2.1 T2V-1.3B: single-GPU vs multi-GPU benchmark

Before running, in the right panel **Settings**:
- **Accelerator → GPU T4 x2**
- **Internet → On** (needs a phone-verified Kaggle account)

Then **Run All**. Total time ≈ 40–70 min, most of it the first model download and generation on T4.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
REPO = "https://github.com/VictorYarosh/wan-video-gpu-benchmark"
!rm -rf /kaggle/working/wan-video-gpu-benchmark
!git clone -q $REPO /kaggle/working/wan-video-gpu-benchmark
%cd /kaggle/working/wan-video-gpu-benchmark

In [ ]:
!pip install -q -U -r requirements.txt
import torch, diffusers, transformers
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__)
print("GPUs:", torch.cuda.device_count())

Download weights once (≈ 17 GB) into `/tmp`, so the download is not counted in the benchmark and does not fill the 20 GB output disk.

In [ ]:
import os
os.environ["HF_HOME"] = "/tmp/hf"
from huggingface_hub import snapshot_download
snapshot_download("Wan-AI/Wan2.1-T2V-1.3B-Diffusers")

## 1. Baseline: one GPU

In [ ]:
!HF_HOME=/tmp/hf python src/bench.py --mode single

## 2. Data parallel: each GPU renders its own video (throughput)

In [ ]:
!HF_HOME=/tmp/hf torchrun --nproc_per_node=2 src/bench.py --mode dp

## 3. Context parallel (Ulysses): two GPUs render ONE video (latency)

Experimental on T4: if diffusers does not support context parallelism with the attention backend available on this GPU, the cell fails. Keep the error text: it is a finding for the README.

In [ ]:
!HF_HOME=/tmp/hf torchrun --nproc_per_node=2 src/bench.py --mode cp

## Results

In [ ]:
!python src/summarize.py results

In [ ]:
from IPython.display import Video, display
import glob
for f in sorted(glob.glob("results/*.mp4")):
    print(f)
    display(Video(f, embed=True, width=480))

In [ ]:
!cd /kaggle/working && zip -qr results.zip wan-video-gpu-benchmark/results
print("Download /kaggle/working/results.zip from the Output panel")